# ECH Deactivation

FResh start!

In [1]:
import os
import pickle
import time  # For debugging query time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location

testDates = [
    "2024-08-08",
    "2024-08-19",
    "2024-08-21",
    "2024-08-29",
    "2024-09-05",
    "2024-09-06",
    "2024-09-08",
    "2024-09-11",
    "2024-09-15",
    "2024-10-13",
    "2024-10-20",
    "2024-10-29",
    "2024-10-31",
    "2024-11-07",
    "2024-11-12",
    "2024-11-15",
    "2024-11-17",
    "2024-11-18",
    "2024-11-19",
    "2024-11-27",
    "2024-11-28",
    "2024-11-29",
    "2024-11-30",
    "2024-12-01",
    "2024-12-02",
    "2024-12-03",
    "2024-12-04",
    "2024-12-12",
    "2024-12-13",
    "2024-12-14",
    "2024-12-15",
    "2024-12-16",
    "2024-12-17",
    "2024-12-18",
    "2024-12-19",
    "2024-12-20",
    "2024-12-21",
    "2024-12-22",
    "2024-12-23",
    "2024-12-24",
    "2024-12-25",
    "2024-12-26",
    "2024-12-29",
    "2024-12-30",
    "2025-01-01",
    "2025-01-02",
    "2025-01-03",
    "2025-01-04",
    "2025-01-05",
    "2025-01-06",
    "2025-01-07",
    "2025-01-08",
    "2025-01-09",
    "2025-01-10",
    "2025-01-11",
    "2025-01-12",
    "2025-01-13",
    "2025-01-14",
    "2025-01-15",
    "2025-01-16",
    "2025-01-17",
    "2025-01-18",
    "2025-01-19",
    "2025-01-20",
    "2025-01-21",
    "2025-01-22",
    "2025-01-23",
    "2025-01-24",
    "2025-01-25",
    "2025-01-26",
    "2025-01-27",
    "2025-01-28",
    "2025-01-29",
    "2025-01-30",
    "2025-01-31",
    "2025-02-01",
    "2025-02-02",
    "2025-02-03",
    "2025-02-04",
    "2025-02-05",
    "2025-02-06",
    "2025-02-07",
    "2025-02-08",
    "2025-02-09",
    "2025-02-10",
    "2025-02-11",
    "2025-02-12",
    "2025-02-14",
    "2025-02-15",
    "2025-02-16",
    "2025-02-17",
    "2025-02-18",
    "2025-02-19",
]


def fetch_and_save_single_date(engine, query, previous_date, current_date, filename_suffix):
    """
    Fetches data for a specific current and previous test_date, executes query,
    and saves the result to a file named with the test_date.
    """
    try:
        print(f"Checking {previous_date} against {current_date}...", flush=True)
        params = (previous_date, current_date)

        # Record the start time
        start_time = time.time()

        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn, params=params)

        # Record the end time
        end_time = time.time()
        print(f"Query complete for {previous_date} against {current_date}.", flush=True)
        print(f"Query execution time: {end_time - start_time:.2f} seconds.", flush=True)

        # Save the data for this test_date to a pickle file
        if df is not None and not df.empty:
            os.makedirs("./pickle", exist_ok=True)
            filename = (
                f"./pickle/{datetime.now().strftime('%Y-%m-%d_%HH-%MM-%SS')}_{current_date}_{filename_suffix}_.pickle"
            )
            with open(filename, "wb") as f:
                pickle.dump(df, f)
            print(f"Data saved to {filename}", flush=True)
        else:
            print(f"No data for {current_date}. No file saved.", flush=True)

        return df
    except Exception as e:
        print(f"Error for {previous_date} against {current_date}: {e}", flush=True)
        return None


def fetch_and_save_data(query, filename_suffix, timeout=3600, max_workers=2):
    """
    Fetches data for multiple test dates, saves separate pickle files for each.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=3,
            max_overflow=10,
        )

        # Create a thread pool to process dates in parallel with limited parallelism
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = []
            for i in range(1, len(testDates)):  # Skip the first date
                previous_date = testDates[i - 1]
                current_date = testDates[i]
                futures.append(
                    executor.submit(
                        fetch_and_save_single_date,
                        engine,
                        query,
                        previous_date,
                        current_date,
                        filename_suffix,
                    )
                )

            # Collect results as they complete
            for future in futures:
                future.result()  # Results are saved in the fetch_and_save_single_date function

    except Exception as e:
        print(f"Error: {e}", flush=True)


if __name__ == "__main__":
    query = """
WITH PreviousSupport AS (
    SELECT
        dr.domain,
        ec.ech_public_name,
        dr.test_date AS previous_test_date
    FROM
        public."DNSResults" dr
    JOIN
        public."DNSResultsECHConfigs" dre ON dr.id = dre.dns_result_id
    JOIN
        public."ECHConfigs" ec ON dre.ech_config_id = ec.id
    WHERE
        dr.https_ech_key != ''
        AND dr.test_date = %s
),
CurrentNoSupport AS (
    SELECT
        dr.domain,
        dr.test_date AS current_test_date
    FROM
        public."DNSResults" dr
    WHERE
        dr.https_ech_key = ''
        AND dr.test_date = %s
)
SELECT
    MAX(cns.current_test_date) AS current_test_date, -- Single test_date
    COUNT(*) AS deactivated_ech_count,               -- Count of deactivated ECH
    ARRAY_AGG(COALESCE(ps.domain, '')) AS previously_supported_domains, -- List of domains
    ARRAY_AGG(COALESCE(ps.ech_public_name, '')) AS ech_public_names     -- List of public names
FROM
    CurrentNoSupport cns
INNER JOIN
    PreviousSupport ps ON cns.domain = ps.domain;
    """
    filename_suffix = "ech_deactivation_by_test_relation_test_dates"
    fetch_and_save_data(query, filename_suffix)

Checking 2024-08-08 against 2024-08-19...
Checking 2024-08-19 against 2024-08-21...
Query complete for 2024-08-19 against 2024-08-21.
Query execution time: 98.51 seconds.
Data saved to ./pickle/2026-04-12_17H-34M-26S_2024-08-21_ech_deactivation_by_test_relation_test_dates_.pickle
Checking 2024-08-21 against 2024-08-29...
Query complete for 2024-08-08 against 2024-08-19.
Query execution time: 109.94 seconds.
Data saved to ./pickle/2026-04-12_17H-34M-38S_2024-08-19_ech_deactivation_by_test_relation_test_dates_.pickle
Checking 2024-08-29 against 2024-09-05...
Query complete for 2024-08-21 against 2024-08-29.
Query execution time: 57.06 seconds.
Data saved to ./pickle/2026-04-12_17H-35M-23S_2024-08-29_ech_deactivation_by_test_relation_test_dates_.pickle
Checking 2024-09-05 against 2024-09-06...
Query complete for 2024-08-29 against 2024-09-05.
Query execution time: 52.18 seconds.
Data saved to ./pickle/2026-04-12_17H-35M-30S_2024-09-05_ech_deactivation_by_test_relation_test_dates_.pickle
C